## Publishing Messages to a Service Bus Queue

### Installing Libraries and Utilities

In [ ]:
%pip install azure-servicebus==7.14.3 openai==2.38.0 python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# loading service bus configurations
service_bus_connection_string = os.getenv("SERVICE_BUS_CONNECTION_STRING")
service_bus_queue_name = os.getenv("SERVICE_BUS_QUEUE_NAME")

# loading azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
chat_completions_model = os.getenv("CHAT_COMPLETIONS_MODEL")

### Creating the Service Bus Client

In [ ]:
from azure.servicebus import ServiceBusClient

sb_client = ServiceBusClient.from_connection_string(
    conn_str = service_bus_connection_string
)

### Sending Payloads to the Queue

In [3]:
questions = [
    {
        "prompt": "Tell me something about Azure Service Bus",
        "model": chat_completions_model
    },
    {
        "prompt": "What are Azure Service Bus Queues?",
        "model": chat_completions_model
    },
    {
        "prompt": "What is GenAI?",
        "model": "claude"
    },
    {
        "prompt": "What are vector embeddings?",
        "model": "deepseek"
    },
    {
        "prompt": "What is Azure Managed Redis?",
        "model": chat_completions_model
    }
]

In [ ]:
from azure.servicebus import ServiceBusMessage
import json
import uuid
from azure.servicebus.exceptions import MessageSizeExceededError

# creating the queue sender object
sender = sb_client.get_queue_sender(service_bus_queue_name)

# creating a counter variable to track the correlation_id value
i=1

for question in questions:
    message = ServiceBusMessage(
        body = json.dumps(question),
        content_type="application/json",
        message_id=str(uuid.uuid4()),
        correlation_id=f"prompt-request-{i}",
        application_properties={
            "request_type": "prompt-request",
            "workload_type": "chat-completions"
        }
    )

    sender.send_messages(message=message)

    i+=1

